# 40. 배포 — 만든 것을 서비스로

> **제40장** · **이론편 대응: 25.6절 (LLM Serving, LLMOps)**
> **예상 소요**: 90분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: **fastapi**, **uvicorn** (1절 참조)
> **다운로드**: 없음

---

## 이 장에서 하는 일

37장에서 **서빙의 원리**를 다뤘다. 이번에는 **실제로 띄운다.**

주피터에서 잘 돌아가는 코드와 **서비스로 쓸 수 있는 코드는 다르다.**

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 준비 — FastAPI 설치 | — |
| 2 | **첫 번째 API** ★ | 25.6절 |
| 3 | 입력 검증과 오류 처리 | 25.6절 |
| 4 | 스트리밍 응답 | 25.6절 |
| 5 | **동시 요청과 비동기** ★ | 25.6절 |
| 6 | 지표 수집 | 25.6절 |
| 7 | Docker 로 묶기 | 25.6절 |
| 8 | 운영 준비 점검 | 25.6절 |

**2절과 5절이 핵심이다.** 주피터 노트북 안에서 실제로 서버를 띄우고 호출해 본다.

---

## 1. 준비 — FastAPI 설치

### 왜 FastAPI인가

```
pip install fastapi uvicorn
```

| 특징 | 설명 |
|---|---|
| **자동 문서화** | `/docs`에서 API 문서를 자동 생성 |
| **입력 검증** | 타입 힌트만 쓰면 검증이 자동 |
| **비동기 지원** | 동시 요청 처리에 유리 (5절) |
| **표준 준수** | OpenAPI 스키마 자동 생성 |

**Flask보다 새롭고, Django보다 가볍다.** LLM 서비스에 널리 쓰인다.

In [ ]:
import importlib

print("=" * 70)
print("필요 패키지 확인")
print("=" * 70)

required = [
    ("fastapi", "웹 프레임워크", "pip install fastapi"),
    ("uvicorn", "ASGI 서버", "pip install uvicorn"),
    ("pydantic", "데이터 검증 (fastapi와 함께 설치)", "pip install pydantic"),
    ("requests", "HTTP 클라이언트 (시험용)", "pip install requests"),
]

missing = []
for name, desc, install in required:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, "__version__", "설치됨")
        print(f"[OK]   {name:<14}{ver:<14}{desc}")
    except ImportError:
        print(f"[없음] {name:<14}{'':<14}{desc}")
        missing.append(install)

print("-" * 70)
if missing:
    print("설치가 필요합니다:")
    for cmd in sorted(set(missing)):
        print(f"  {cmd}")
else:
    print("[준비 완료] 2절로 진행하세요.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
import time
import json
import threading
from pathlib import Path

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent

print("준비 완료")
print()
print("[이 장의 방식]")
print("  주피터 노트북 안에서 서버를 띄워 실제로 호출해 본다.")
print("  실무에서는 별도 파일(main.py)로 만들어 터미널에서 실행한다.")

---

## 2. 첫 번째 API ★ — 이론편 25.6절

**43장의 문서 QA 시스템을 API로 만든다.**

앞 장에서 함수로 부르던 것을 **HTTP 요청으로** 부를 수 있게 하는 것이다.

In [ ]:
import time
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel, Field

# ── 간이 QA 시스템 (43장의 축약판) ──
DOCUMENTS = [
    "재택근무는 주 2회까지 신청 가능하며 팀장 승인이 필요합니다.",
    "연차는 입사 1년 미만은 월 1일, 1년 이상은 연 15일이 부여됩니다.",
    "국내 출장비는 일 8만원, 해외는 일 15만원까지 정산 가능합니다.",
    "교육비는 연간 200만원까지 지원되며 업무 관련성이 인정되어야 합니다.",
    "노트북은 3년마다 교체 대상이 되며 고장 시 즉시 신청 가능합니다.",
]

KEYWORDS = {
    0: ["재택", "집", "원격"],
    1: ["연차", "휴가", "쉬"],
    2: ["출장", "여비", "해외"],
    3: ["교육", "공부", "학습"],
    4: ["노트북", "컴퓨터", "장비"],
}


def simple_search(question, top_k=3):
    """간단한 키워드 검색 (실제로는 27장의 임베딩 검색)"""
    scores = []
    for idx, keywords in KEYWORDS.items():
        score = sum(1 for kw in keywords if kw in question)
        scores.append((idx, score))
    scores.sort(key=lambda x: -x[1])
    return [(i, s) for i, s in scores[:top_k] if s > 0]


def answer_question(question, top_k=3):
    """질문에 답한다"""
    hits = simple_search(question, top_k)
    if not hits:
        return {
            "answer": "관련 규정을 찾을 수 없습니다. 담당 부서에 문의해 주세요.",
            "sources": [],
            "confidence": 0.0,
        }
    return {
        "answer": DOCUMENTS[hits[0][0]],
        "sources": [{"id": i, "text": DOCUMENTS[i][:40], "score": float(s)}
                    for i, s in hits],
        "confidence": float(hits[0][1]) / 3,
    }


print("=" * 78)
print("QA 시스템 동작 확인 (API 로 감싸기 전)")
print("=" * 78)
for q in ["재택근무 규정", "연차 며칠", "주차장 이용"]:
    result = answer_question(q)
    print(f"\n질문: {q}")
    print(f"  답변: {result['answer'][:44]}")
    print(f"  신뢰도: {result['confidence']:.2f}")

In [ ]:
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional
import time

# ── 요청·응답 형식 정의 ──
class QuestionRequest(BaseModel):
    """요청 형식 — 타입 힌트만으로 검증이 자동으로 된다"""
    question: str = Field(..., min_length=1, max_length=500,
                          description="질문 내용")
    top_k: int = Field(3, ge=1, le=10, description="검색할 문서 수")


class Source(BaseModel):
    id: int
    text: str
    score: float


class AnswerResponse(BaseModel):
    """응답 형식"""
    answer: str
    sources: List[Source]
    confidence: float
    latency_ms: float


# ── 앱 만들기 ──
app = FastAPI(
    title="사내 문서 QA API",
    description="사내 규정을 검색해 답변합니다",
    version="1.0.0",
)


@app.get("/health")
def health_check():
    """상태 확인 — 로드밸런서가 주기적으로 호출한다"""
    return {"status": "ok", "version": "1.0.0"}


@app.post("/answer", response_model=AnswerResponse)
def get_answer(request: QuestionRequest):
    """질문에 답한다"""
    t0 = time.time()
    result = answer_question(request.question, request.top_k)
    return AnswerResponse(
        answer=result["answer"],
        sources=result["sources"],
        confidence=result["confidence"],
        latency_ms=(time.time() - t0) * 1000,
    )


print("=" * 78)
print("API 정의 완료")
print("=" * 78)
print()
print("정의한 엔드포인트")
for route in app.routes:
    if hasattr(route, "methods") and hasattr(route, "path"):
        methods = ",".join(sorted(route.methods - {"HEAD", "OPTIONS"}))
        if methods:
            print(f"  {methods:<8}{route.path}")
print()
print("[Pydantic 모델의 역할]")
print("  QuestionRequest : 들어오는 요청을 검증한다")
print("  AnswerResponse  : 나가는 응답 형식을 보장한다")
print()
print("  타입에 맞지 않으면 자동으로 422 오류를 돌려준다 (3절).")

In [ ]:
import warnings
from fastapi.testclient import TestClient

print("=" * 78)
print("TestClient 로 호출해 보기")
print("=" * 78)
print("(서버를 띄우지 않고 앱을 직접 호출한다 — 시험에 편리)")
print()

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    client = TestClient(app)

# 상태 확인
response = client.get("/health")
print(f"GET /health")
print(f"  상태 코드: {response.status_code}")
print(f"  응답     : {response.json()}")
print()

# 질문
response = client.post("/answer", json={"question": "재택근무는 어떻게 신청하나요?"})
print(f"POST /answer")
print(f"  상태 코드: {response.status_code}")
data = response.json()
print(f"  답변     : {data['answer'][:44]}")
print(f"  신뢰도   : {data['confidence']:.2f}")
print(f"  지연     : {data['latency_ms']:.2f} ms")
print(f"  근거     : {len(data['sources'])}개")
print()

# 여러 질문
print("여러 질문 처리")
print(f"{'질문':<26}{'상태':<10}{'신뢰도':<12}{'답변'}")
print("-" * 78)
for q in ["연차 규정", "출장비 한도", "주차장 위치"]:
    r = client.post("/answer", json={"question": q})
    d = r.json()
    print(f"{q:<26}{r.status_code:<10}{d['confidence']:<12.2f}{d['answer'][:26]}")
print("-" * 78)

In [ ]:
import warnings
from fastapi.testclient import TestClient
import json

print("=" * 78)
print("자동 생성된 API 문서")
print("=" * 78)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    client = TestClient(app)

response = client.get("/openapi.json")
schema = response.json()

print(f"제목  : {schema['info']['title']}")
print(f"버전  : {schema['info']['version']}")
print(f"설명  : {schema['info']['description']}")
print()
print("경로")
for path, methods in schema["paths"].items():
    for method, spec in methods.items():
        print(f"  {method.upper():<8}{path:<16}{spec.get('summary', '')}")
print()

print("요청 스키마 (QuestionRequest)")
req_schema = schema["components"]["schemas"]["QuestionRequest"]
for field, spec in req_schema["properties"].items():
    constraints = []
    for key in ["minLength", "maxLength", "minimum", "maximum", "default"]:
        if key in spec:
            constraints.append(f"{key}={spec[key]}")
    print(f"  {field:<14}{spec.get('type', '?'):<12}{', '.join(constraints)}")
print()
print("-" * 78)
print("[문서를 따로 쓰지 않아도 된다]")
print("  서버를 띄우면 http://localhost:8000/docs 에서")
print("  브라우저로 API 를 시험해 볼 수 있는 화면이 자동 생성된다.")
print()
print("  프론트엔드 개발자에게 이 주소만 알려주면 된다.")

---

## 3. 입력 검증과 오류 처리 — 이론편 25.6절

**사용자는 예상하지 못한 값을 보낸다.** 서비스는 그것을 견뎌야 한다.

FastAPI는 Pydantic 덕분에 **검증이 자동**이다.

In [ ]:
import warnings
from fastapi.testclient import TestClient

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    client = TestClient(app)

print("=" * 78)
print("잘못된 요청 처리")
print("=" * 78)
print()

bad_requests = [
    ({"question": ""},                      "빈 질문 (min_length=1)"),
    ({"question": "x" * 600},               "너무 김 (max_length=500)"),
    ({"question": "test", "top_k": 99},     "top_k 범위 초과 (le=10)"),
    ({"question": "test", "top_k": 0},      "top_k 범위 미만 (ge=1)"),
    ({"top_k": 3},                          "필수 필드 누락"),
    ({"question": 12345},                   "타입 불일치"),
]

print(f"{'요청':<40}{'상태':<10}{'설명'}")
print("-" * 78)
for payload, desc in bad_requests:
    r = client.post("/answer", json=payload)
    preview = str(payload)[:36]
    print(f"{preview:<40}{r.status_code:<10}{desc}")
print("-" * 78)
print()

# 오류 응답 내용 확인
r = client.post("/answer", json={"question": "", "top_k": 99})
print("오류 응답 예시 (422)")
detail = r.json().get("detail", [])
for item in detail[:3]:
    loc = " → ".join(str(x) for x in item.get("loc", []))
    print(f"  위치: {loc}")
    print(f"  내용: {item.get('msg', '')}")
print()
print("[자동 검증의 이점]")
print("  검증 코드를 직접 쓰지 않아도 된다.")
print("  오류 메시지가 일관된 형식으로 나간다.")
print("  어느 필드가 왜 잘못됐는지 클라이언트가 알 수 있다.")

In [ ]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
from pydantic import BaseModel, Field
import time
import logging

print("=" * 78)
print("직접 만드는 오류 처리")
print("=" * 78)

# 로깅 설정
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger("qa-api")

# httpx 의 요청 로그가 너무 많아 억제한다 (주피터 실행용)
logging.getLogger("httpx").setLevel(logging.WARNING)

app_v2 = FastAPI(title="QA API v2")

MIN_CONFIDENCE = 0.3


class QuestionRequestV2(BaseModel):
    question: str = Field(..., min_length=1, max_length=500)
    top_k: int = Field(3, ge=1, le=10)


@app_v2.get("/health")
def health():
    return {"status": "ok"}


@app_v2.post("/answer")
def answer_v2(request: QuestionRequestV2):
    """신뢰도가 낮으면 명시적으로 알린다"""
    t0 = time.time()

    try:
        result = answer_question(request.question, request.top_k)
    except Exception as e:
        # 내부 오류는 로그에 남기고, 사용자에겐 일반적인 메시지만
        logger.error(f"처리 실패: {type(e).__name__}: {e}")
        raise HTTPException(status_code=500,
                            detail="일시적인 오류가 발생했습니다.")

    if result["confidence"] < MIN_CONFIDENCE:
        # 404 로 '찾을 수 없음'을 명시한다
        raise HTTPException(
            status_code=404,
            detail={
                "message": "관련 규정을 찾을 수 없습니다.",
                "suggestion": "다른 표현으로 질문해 보시거나 담당 부서에 문의해 주세요.",
                "confidence": result["confidence"],
            })

    return {
        **result,
        "latency_ms": (time.time() - t0) * 1000,
    }


@app_v2.exception_handler(Exception)
async def general_exception_handler(request: Request, exc: Exception):
    """예상치 못한 오류를 모두 잡는다"""
    logger.error(f"처리되지 않은 오류: {type(exc).__name__}: {exc}")
    return JSONResponse(
        status_code=500,
        content={"detail": "서버 오류가 발생했습니다."},
    )


import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    client_v2 = TestClient(app_v2, raise_server_exceptions=False)

print()
print(f"{'질문':<26}{'상태':<10}{'응답'}")
print("-" * 78)
for q in ["재택근무 신청", "연차 규정", "주차장 이용 방법"]:
    r = client_v2.post("/answer", json={"question": q})
    if r.status_code == 200:
        preview = r.json()["answer"][:32]
    else:
        d = r.json().get("detail", {})
        preview = d.get("message", str(d))[:32] if isinstance(d, dict) else str(d)[:32]
    print(f"{q:<26}{r.status_code:<10}{preview}")
print("-" * 78)
print()
print("[상태 코드를 제대로 쓰자]")
print(f"  {'200':<8}정상 처리")
print(f"  {'404':<8}찾을 수 없음 (신뢰도 미달)")
print(f"  {'422':<8}요청 형식 오류 (자동)")
print(f"  {'429':<8}요청 한도 초과")
print(f"  {'500':<8}서버 내부 오류")
print(f"  {'503':<8}일시적 과부하")
print()
print("[중요] 내부 오류 메시지를 사용자에게 노출하지 말 것")
print("  스택 트레이스에는 파일 경로·변수명 등이 담긴다.")
print("  로그에는 자세히, 응답에는 일반적으로.")

---

## 4. 스트리밍 응답 — 이론편 25.6절

41장 1절에서 **TTFT(첫 토큰까지의 시간)**를 다뤘다.

전체 응답을 기다리게 하는 대신 **생성되는 대로 보내면** 체감 속도가 크게 달라진다.

In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
import time
import json
import asyncio

app_stream = FastAPI(title="QA API — 스트리밍")


class StreamRequest(BaseModel):
    question: str


def generate_tokens(text, delay=0.05):
    """토큰을 하나씩 내보낸다 (실제로는 모델 생성)"""
    for word in text.split():
        yield word + " "
        time.sleep(delay)


@app_stream.post("/answer/stream")
def stream_answer(request: StreamRequest):
    """Server-Sent Events 형식으로 스트리밍"""
    result = answer_question(request.question)

    def event_generator():
        # 먼저 메타데이터를 보낸다
        meta = {"type": "meta", "sources": len(result["sources"]),
                "confidence": result["confidence"]}
        yield f"data: {json.dumps(meta, ensure_ascii=False)}\n\n"

        # 그다음 본문을 조각내어 보낸다
        for chunk in generate_tokens(result["answer"], delay=0.03):
            payload = {"type": "token", "content": chunk}
            yield f"data: {json.dumps(payload, ensure_ascii=False)}\n\n"

        yield 'data: {"type": "done"}\n\n'

    return StreamingResponse(event_generator(), media_type="text/event-stream")


print("=" * 78)
print("스트리밍 응답 확인")
print("=" * 78)

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    client_s = TestClient(app_stream)

t0 = time.time()
with client_s.stream("POST", "/answer/stream",
                     json={"question": "재택근무 규정"}) as response:
    first_token_time = None
    tokens = []
    for line in response.iter_lines():
        if not line or not line.startswith("data: "):
            continue
        payload = json.loads(line[6:])
        if payload["type"] == "meta":
            print(f"  [메타] 근거 {payload['sources']}개, "
                  f"신뢰도 {payload['confidence']:.2f}  "
                  f"({(time.time()-t0)*1000:.0f}ms)")
        elif payload["type"] == "token":
            if first_token_time is None:
                first_token_time = time.time() - t0
                print(f"  [첫 토큰] {(first_token_time)*1000:.0f}ms")
            tokens.append(payload["content"])
        elif payload["type"] == "done":
            total = time.time() - t0
            print(f"  [완료] {total*1000:.0f}ms")

print()
print(f"받은 내용: {''.join(tokens)[:50]}...")
print()
print(f"{'지표':<20}{'값'}")
print("-" * 78)
print(f"{'TTFT':<20}{first_token_time*1000:.0f} ms")
print(f"{'전체 시간':<20}{total*1000:.0f} ms")
print(f"{'비율':<20}{first_token_time/total*100:.0f}%")
print("-" * 78)
print()
print("[체감의 차이]")
print("  전체를 기다리면 사용자는 아무것도 못 본 채 기다린다.")
print("  스트리밍하면 첫 단어부터 읽기 시작할 수 있다.")
print()
print("  전체 시간은 같아도 **체감 속도가 크게 다르다** (41장 1절).")

---

## 5. 동시 요청과 비동기 ★ — 이론편 25.6절

**서비스에는 여러 사람이 동시에 접속한다.**

41장에서 배치 처리를 다뤘다. 여기서는 **웹 서버 수준의 동시성**을 본다.

| 방식 | 동작 | 적합한 작업 |
|---|---|---|
| `def` (동기) | 스레드풀에서 실행 | CPU 작업, 블로킹 I/O |
| `async def` (비동기) | 이벤트 루프에서 실행 | **네트워크 대기** |

**LLM API를 호출하는 서비스라면 비동기가 유리하다** — 대부분의 시간을 기다리기 때문이다.

In [ ]:
import asyncio
import time
from fastapi import FastAPI

app_async = FastAPI(title="동기 vs 비동기")

WAIT_TIME = 0.3      # 외부 API 호출을 흉내


@app_async.get("/sync")
def sync_endpoint():
    """동기 — 스레드가 블로킹된다"""
    time.sleep(WAIT_TIME)
    return {"mode": "sync"}


@app_async.get("/async")
async def async_endpoint():
    """비동기 — 기다리는 동안 다른 요청을 처리한다"""
    await asyncio.sleep(WAIT_TIME)
    return {"mode": "async"}


print("=" * 78)
print("동기 vs 비동기 — 무엇이 다른가")
print("=" * 78)
print()
print("코드")
print()
print("  # 동기 — 잘못된 예 (async 안에서 블로킹)")
print("  @app.get('/bad')")
print("  async def bad():")
print("      time.sleep(1)          # ← 이벤트 루프 전체가 멈춘다!")
print()
print("  # 동기 — 올바른 예")
print("  @app.get('/ok')")
print("  def ok():")
print("      time.sleep(1)          # ← 스레드풀에서 실행되므로 괜찮다")
print()
print("  # 비동기 — 가장 좋음")
print("  @app.get('/best')")
print("  async def best():")
print("      await asyncio.sleep(1) # ← 기다리는 동안 다른 요청 처리")
print()
print("-" * 78)
print("[가장 흔한 실수]")
print("  async def 안에서 time.sleep() 이나 requests.get() 을 쓰는 것")
print("  → 이벤트 루프가 멈춰 동시성이 사라진다")
print()
print("  비동기 함수 안에서는 await 가능한 것만 써야 한다:")
print("    time.sleep()   → await asyncio.sleep()")
print("    requests.get() → await httpx.AsyncClient().get()")

In [ ]:
import threading
import time
import requests
import uvicorn
import numpy as np
from concurrent.futures import ThreadPoolExecutor

print("=" * 78)
print("실제 서버를 띄워 동시 요청 시험")
print("=" * 78)

PORT = 8765

def run_server():
    uvicorn.run(app_async, host="127.0.0.1", port=PORT, log_level="error")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

# 서버가 뜰 때까지 기다린다
base_url = f"http://127.0.0.1:{PORT}"
for _ in range(20):
    try:
        requests.get(f"{base_url}/async", timeout=1)
        break
    except Exception:
        time.sleep(0.5)

print(f"서버 시작: {base_url}")
print()


def send_requests(path, n_concurrent):
    """동시에 n개 요청을 보낸다"""
    def one():
        t0 = time.time()
        try:
            requests.get(f"{base_url}{path}", timeout=30)
        except Exception:
            pass
        return time.time() - t0

    t_start = time.time()
    with ThreadPoolExecutor(max_workers=n_concurrent) as pool:
        latencies = list(pool.map(lambda _: one(), range(n_concurrent)))
    total = time.time() - t_start
    return total, latencies


print(f"각 요청은 {WAIT_TIME}초를 기다린다")
print()
print(f"{'동시 요청':<14}{'동기 전체':<16}{'비동기 전체':<16}{'비동기 이득'}")
print("-" * 78)

comparison = []
for n in [1, 5, 10, 20]:
    total_sync, lat_sync = send_requests("/sync", n)
    total_async, lat_async = send_requests("/async", n)
    comparison.append({
        "n": n, "sync": total_sync, "async": total_async,
        "sync_p95": np.percentile(lat_sync, 95),
        "async_p95": np.percentile(lat_async, 95),
    })
    speedup = total_sync / total_async if total_async > 0 else 1
    print(f"{n:<14}{total_sync:<16.2f}{total_async:<16.2f}{speedup:.2f}배")

print("-" * 78)
print()
print("[해석]")
print("  요청이 늘수록 비동기의 이득이 커진다.")
print("  동기는 스레드풀 크기에 제한되지만, 비동기는 대기 중에 다른 일을 한다.")
print()
print("  실제 LLM 서비스는 대부분의 시간을 '모델 응답 대기'로 보낸다.")
print("  → 비동기가 특히 유리하다")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

ns = [c["n"] for c in comparison]
syncs = [c["sync"] for c in comparison]
asyncs = [c["async"] for c in comparison]

# --- 왼쪽: 전체 처리 시간 ---
ax = axes[0]
ax.plot(ns, syncs, marker="o", linewidth=2.5, color="#DC2626", label="동기")
ax.plot(ns, asyncs, marker="s", linewidth=2.5, color="#0D9488", label="비동기")
ax.axhline(WAIT_TIME, color="gray", linestyle="--", linewidth=1.5)
ax.text(ns[-1] * 0.55, WAIT_TIME * 1.15, f"이론 최솟값 {WAIT_TIME}초",
        fontsize=8, color="gray")
ax.set_xlabel("동시 요청 수")
ax.set_ylabel("전체 처리 시간 (초)")
ax.set_title("동시 요청 처리 시간")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# --- 오른쪽: p95 지연 ---
ax = axes[1]
x = np.arange(len(ns))
w = 0.36
ax.bar(x - w/2, [c["sync_p95"] for c in comparison], w,
       label="동기 p95", color="#DC2626")
ax.bar(x + w/2, [c["async_p95"] for c in comparison], w,
       label="비동기 p95", color="#0D9488")
ax.set_xticks(x)
ax.set_xticklabels([str(n) for n in ns])
ax.set_xlabel("동시 요청 수")
ax.set_ylabel("p95 지연 (초)")
ax.set_title("개별 요청의 대기 시간 (37장 6절)")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 78)
print("워커 수 설정")
print("=" * 78)
print()
print("  uvicorn main:app --workers 4")
print()
print(f"{'작업 성격':<26}{'권장 설정':<26}{'이유'}")
print("-" * 78)
print(f"{'CPU 집약 (로컬 모델)':<26}{'workers = CPU 코어 수':<26}{'병렬 계산'}")
print(f"{'I/O 대기 (외부 API)':<26}{'async + workers 2~4':<26}{'대기 중 다른 일'}")
print(f"{'혼합':<26}{'측정해서 결정':<26}{'—'}")
print("-" * 78)
print()
print("[주의] 로컬 모델을 쓸 때")
print("  워커마다 모델을 따로 올리면 메모리가 배수로 든다.")
print("  워커 4개 x 7B 모델 = 메모리 4배")
print()
print("  → 모델 서버(vLLM 등)를 따로 두고 API 는 호출만 하는 구조가 낫다")

---

## 6. 지표 수집 — 이론편 25.6절

41장에서 **무엇을 측정할지** 다뤘다. 이번에는 **어떻게 수집할지**를 본다.

In [ ]:
from fastapi import FastAPI, Request
from pydantic import BaseModel, Field
import time
import numpy as np
from collections import defaultdict, deque

app_metrics = FastAPI(title="QA API — 지표 수집")


class MetricsCollector:
    """요청 지표를 모은다 (37장 6절)"""

    def __init__(self, window=1000):
        self.latencies = deque(maxlen=window)
        self.status_counts = defaultdict(int)
        self.endpoint_counts = defaultdict(int)
        self.start_time = time.time()

    def record(self, path, status, latency):
        self.latencies.append(latency)
        self.status_counts[status] += 1
        self.endpoint_counts[path] += 1

    def summary(self):
        if not self.latencies:
            return {"requests": 0}
        lat = np.array(self.latencies)
        uptime = time.time() - self.start_time
        total = sum(self.status_counts.values())
        errors = sum(v for k, v in self.status_counts.items() if k >= 400)
        return {
            "requests": total,
            "uptime_sec": round(uptime, 1),
            "rps": round(total / uptime, 2) if uptime > 0 else 0,
            "latency_mean_ms": round(float(lat.mean()) * 1000, 1),
            "latency_p50_ms": round(float(np.percentile(lat, 50)) * 1000, 1),
            "latency_p95_ms": round(float(np.percentile(lat, 95)) * 1000, 1),
            "latency_p99_ms": round(float(np.percentile(lat, 99)) * 1000, 1),
            "error_rate": round(errors / total, 4) if total else 0,
            "status_counts": dict(self.status_counts),
        }


metrics = MetricsCollector()


@app_metrics.middleware("http")
async def track_metrics(request: Request, call_next):
    """모든 요청을 가로채 지표를 기록한다"""
    t0 = time.time()
    response = await call_next(request)
    latency = time.time() - t0

    metrics.record(request.url.path, response.status_code, latency)
    # 응답 헤더에 처리 시간을 넣는다 (디버깅에 유용)
    response.headers["X-Process-Time"] = f"{latency*1000:.2f}"
    return response


class QReq(BaseModel):
    question: str = Field(..., min_length=1, max_length=500)


@app_metrics.get("/health")
def health_m():
    return {"status": "ok"}


@app_metrics.post("/answer")
def answer_m(req: QReq):
    return answer_question(req.question)


@app_metrics.get("/metrics")
def get_metrics():
    """지표 조회 — 모니터링 도구가 주기적으로 호출한다"""
    return metrics.summary()


print("=" * 78)
print("미들웨어로 지표 수집")
print("=" * 78)
print()
print("[미들웨어란]")
print("  모든 요청이 지나가는 통로다.")
print("  각 엔드포인트에 코드를 넣지 않아도 일괄 처리할 수 있다.")
print()
print("  용도: 지표 수집, 인증, 로깅, CORS, 요청 ID 부여 등")

In [ ]:
import warnings
from fastapi.testclient import TestClient
import numpy as np
import json

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    client_m = TestClient(app_metrics)

print("=" * 78)
print("요청을 보내 지표 쌓기")
print("=" * 78)

questions = ["재택근무 규정", "연차 며칠", "출장비 한도",
             "교육비 지원", "노트북 교체", "주차장 위치"]

for _ in range(5):
    for q in questions:
        client_m.post("/answer", json={"question": q})
    client_m.get("/health")
    client_m.post("/answer", json={"question": ""})      # 오류 발생

print(f"총 {sum(metrics.status_counts.values())}건 처리")
print()

response = client_m.get("/metrics")
summary = response.json()

print("수집된 지표")
print("-" * 78)
for key, value in summary.items():
    if key != "status_counts":
        print(f"  {key:<22}{value}")
print()
print("  상태 코드별")
for code, count in sorted(summary["status_counts"].items()):
    print(f"    {code}: {count}건")
print("-" * 78)
print()

# 응답 헤더 확인
r = client_m.post("/answer", json={"question": "연차 규정"})
print(f"응답 헤더 X-Process-Time: {r.headers.get('X-Process-Time')} ms")
print()
print("[p50 vs p95 vs p99]")
print(f"  p50: {summary['latency_p50_ms']} ms  — 절반은 이보다 빠르다")
print(f"  p95: {summary['latency_p95_ms']} ms  — 20명 중 1명이 겪는 시간")
print(f"  p99: {summary['latency_p99_ms']} ms  — 100명 중 1명")
print()
print("  41장 6절에서 강조했듯 **평균이 아니라 p95 를 본다**.")

In [ ]:
print("=" * 78)
print("모니터링 도구와 연동")
print("=" * 78)
print()
print("[Prometheus 형식으로 내보내기]")
print()
prom_example = [
    "from prometheus_client import Counter, Histogram, generate_latest",
    "from fastapi import Response",
    "",
    "REQUEST_COUNT = Counter('http_requests_total',",
    "                        'Total requests', ['method', 'path', 'status'])",
    "REQUEST_LATENCY = Histogram('http_request_duration_seconds',",
    "                            'Request latency', ['path'])",
    "",
    "@app.middleware('http')",
    "async def track(request, call_next):",
    "    with REQUEST_LATENCY.labels(request.url.path).time():",
    "        response = await call_next(request)",
    "    REQUEST_COUNT.labels(request.method, request.url.path,",
    "                         response.status_code).inc()",
    "    return response",
    "",
    "@app.get('/metrics')",
    "def metrics():",
    "    return Response(generate_latest(), media_type='text/plain')",
]
for line in prom_example:
    print("  " + line)

print()
print("-" * 78)
print("[LLM 서비스에서 추가로 볼 것]")
print()
print(f"{'지표':<26}{'왜':<30}{'수집 방법'}")
print("-" * 78)
llm_metrics = [
    ("TTFT", "체감 속도 (37장 1절)", "첫 토큰 시각 기록"),
    ("입력·출력 토큰 수", "비용 계산 (25장)", "응답에서 usage 추출"),
    ("검색 신뢰도 분포", "RAG 품질 (43장 8절)", "요청마다 기록"),
    ("거절률", "문서 부족 신호", "404 비율"),
    ("재질문률", "답변 품질 신호", "세션 단위 추적"),
    ("모델·프롬프트 버전", "재현성 (37장 8절)", "응답에 포함"),
]
for a, b, c in llm_metrics:
    print(f"{a:<26}{b:<30}{c}")
print("-" * 78)

---

## 7. Docker로 묶기 — 이론편 25.6절

**"내 컴퓨터에서는 되는데"를 없앤다.**

코드·의존성·설정을 하나로 묶어 어디서든 같게 실행되게 한다.

In [ ]:
from pathlib import Path

root = Path.cwd()
if root.name.startswith("part"):
    root = root.parent
deploy_dir = root / "outputs" / "deploy"
deploy_dir.mkdir(parents=True, exist_ok=True)

# ── main.py ──
main_py = '''"""사내 문서 QA API

실행:
    uvicorn main:app --host 0.0.0.0 --port 8000
"""
import time
import logging
from typing import List

from fastapi import FastAPI, HTTPException, Request
from pydantic import BaseModel, Field

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI(title="사내 문서 QA API", version="1.0.0")

MIN_CONFIDENCE = 0.3

# 실제로는 여기서 모델과 인덱스를 불러온다
DOCUMENTS = [
    "재택근무는 주 2회까지 신청 가능하며 팀장 승인이 필요합니다.",
    "연차는 입사 1년 미만은 월 1일, 1년 이상은 연 15일이 부여됩니다.",
]


class QuestionRequest(BaseModel):
    question: str = Field(..., min_length=1, max_length=500)
    top_k: int = Field(3, ge=1, le=10)


@app.on_event("startup")
async def startup():
    """서버 시작 시 한 번 — 모델 로딩을 여기서"""
    logger.info("서비스 시작")


@app.get("/health")
def health():
    return {"status": "ok", "version": app.version}


@app.post("/answer")
def answer(request: QuestionRequest):
    t0 = time.time()
    # 실제 검색·생성 로직
    result = {"answer": DOCUMENTS[0], "sources": [], "confidence": 1.0}

    if result["confidence"] < MIN_CONFIDENCE:
        raise HTTPException(404, "관련 규정을 찾을 수 없습니다.")

    return {**result, "latency_ms": (time.time() - t0) * 1000}
'''

(deploy_dir / "main.py").write_text(main_py, encoding="utf-8")

# ── requirements.txt ──
requirements = """fastapi>=0.100.0
uvicorn[standard]>=0.23.0
pydantic>=2.0.0
sentence-transformers>=2.2.0
"""
(deploy_dir / "requirements.txt").write_text(requirements, encoding="utf-8")

# ── Dockerfile ──
dockerfile = """# 가벼운 기본 이미지
FROM python:3.13-slim

WORKDIR /app

# 1) 의존성을 먼저 복사해 설치 — 캐시 활용
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# 2) 코드는 나중에 복사 — 코드만 바뀌면 위 단계는 재사용
COPY . .

# 3) 모델을 미리 받아 둔다 (선택)
# RUN python -c "from sentence_transformers import SentenceTransformer; \\
#     SentenceTransformer('모델명')"

EXPOSE 8000

# 4) 상태 확인
HEALTHCHECK --interval=30s --timeout=3s --start-period=40s \\
    CMD python -c "import urllib.request; \\
    urllib.request.urlopen('http://localhost:8000/health')"

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
"""
(deploy_dir / "Dockerfile").write_text(dockerfile, encoding="utf-8")

# ── docker-compose.yml ──
compose = """services:
  qa-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - OPENAI_API_KEY=${OPENAI_API_KEY}
      - LOG_LEVEL=info
    volumes:
      # 모델 캐시를 공유해 재시작 시 다시 받지 않게
      - ./model_cache:/root/.cache/huggingface
    restart: unless-stopped
    deploy:
      resources:
        limits:
          memory: 4G
"""
(deploy_dir / "docker-compose.yml").write_text(compose, encoding="utf-8")

# ── .dockerignore ──
dockerignore = """__pycache__/
*.pyc
.env
.git/
data/
outputs/
*.ipynb
"""
(deploy_dir / ".dockerignore").write_text(dockerignore, encoding="utf-8")

print("=" * 78)
print("배포 파일 생성")
print("=" * 78)
print(f"위치: {deploy_dir}")
print()
for f in sorted(deploy_dir.iterdir()):
    if f.is_file():
        print(f"  {f.name:<26}{f.stat().st_size:>8,} 바이트")
print()
print("Dockerfile 내용")
print("-" * 78)
print(dockerfile)
print("-" * 78)

In [ ]:
print("=" * 78)
print("Dockerfile 작성 요령")
print("=" * 78)
print()
print(f"{'요령':<30}{'이유'}")
print("-" * 78)
tips = [
    ("slim 이미지 사용", "전체 이미지는 1GB 넘음, slim 은 150MB"),
    ("requirements 먼저 COPY", "코드만 바뀌면 설치를 건너뛴다 (캐시)"),
    ("--no-cache-dir", "pip 캐시를 남기지 않아 이미지가 작아짐"),
    (".dockerignore 작성", "불필요한 파일이 이미지에 들어가지 않게"),
    ("HEALTHCHECK 추가", "컨테이너 상태를 자동 감시"),
    ("모델 미리 받기", "첫 요청이 느려지지 않게 (선택)"),
    ("비밀은 환경변수로", "이미지에 API 키를 넣지 말 것"),
]
for a, b in tips:
    print(f"{a:<30}{b}")
print("-" * 78)
print()

print("실행 명령")
print()
commands = [
    "# 이미지 빌드",
    "docker build -t qa-api:1.0 .",
    "",
    "# 실행",
    "docker run -p 8000:8000 --env-file .env qa-api:1.0",
    "",
    "# 또는 compose 로",
    "docker compose up -d",
    "",
    "# 로그 확인",
    "docker compose logs -f",
    "",
    "# 중지",
    "docker compose down",
]
for c in commands:
    print("  " + c)

print()
print("-" * 78)
print("[LLM 서비스의 특수한 점]")
print()
print(f"{'문제':<26}{'대응'}")
print("-" * 78)
print(f"{'모델 파일이 큼 (GB 단위)':<26}볼륨으로 마운트하거나 시작 시 다운로드")
print(f"{'첫 요청이 매우 느림':<26}startup 에서 미리 로딩 + 워밍업 요청")
print(f"{'메모리 사용량이 큼':<26}컨테이너 메모리 한도 설정 필수")
print(f"{'GPU 사용':<26}nvidia-docker 또는 --gpus all")
print("-" * 78)

---

## 8. 운영 준비 점검 — 이론편 25.6절

이 장의 체크리스트를 **배포 관점에서** 구체화한다.

In [ ]:
print("=" * 78)
print("배포 전 최종 점검")
print("=" * 78)
print()

checklist = {
    "API 설계": [
        "상태 확인 엔드포인트(/health)가 있는가",
        "입력 길이·범위 제한이 있는가 (3절)",
        "오류 상태 코드를 구분해 쓰는가",
        "내부 오류 메시지가 노출되지 않는가",
        "API 버전을 관리하는가 (/v1/answer)",
    ],
    "성능": [
        "타임아웃을 설정했는가",
        "동시 요청 수를 제한했는가",
        "비동기가 필요한 부분에 적용했는가 (5절)",
        "부하 시험을 했는가 (예상 최대의 2배)",
        "p95 지연이 목표 안에 드는가",
    ],
    "안전": [
        "API 키가 코드나 이미지에 없는가",
        "요청 한도(rate limit)가 있는가",
        "프롬프트 주입 대비가 있는가 (37장 7절)",
        "CORS 설정이 적절한가",
        "HTTPS 를 쓰는가",
    ],
    "관측": [
        "요청 지표를 수집하는가 (6절)",
        "로그가 남는가",
        "요청 추적 ID 가 있는가",
        "이상 상황 알림이 있는가",
        "모델·프롬프트 버전을 기록하는가 (41장 8절)",
    ],
    "배포": [
        "Dockerfile 이 있는가 (7절)",
        "환경변수로 설정을 주입하는가",
        "이전 버전으로 롤백할 수 있는가",
        "무중단 배포가 가능한가",
        "회귀 시험이 자동으로 도는가 (39장 7절)",
    ],
}

total = 0
for category, items in checklist.items():
    print(f"[{category}]")
    for item in items:
        print(f"  □ {item}")
    total += len(items)
    print()

print("-" * 78)
print(f"총 {total}개 항목")
print()
print("모두 갖추고 시작할 필요는 없다.")
print("다만 **무엇이 없는지는 알고** 시작해야 한다.")

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

print("=" * 78)
print("간이 부하 시험")
print("=" * 78)
print()

import warnings
from fastapi.testclient import TestClient

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    client_load = TestClient(app)

test_questions = ["재택근무", "연차", "출장비", "교육비", "노트북"]

n_requests = 200
latencies = []
statuses = []

t_start = time.time()
for i in range(n_requests):
    q = test_questions[i % len(test_questions)]
    t0 = time.time()
    r = client_load.post("/answer", json={"question": q})
    latencies.append((time.time() - t0) * 1000)
    statuses.append(r.status_code)
total_time = time.time() - t_start

lat = np.array(latencies)
print(f"요청 {n_requests}건, {total_time:.2f}초")
print()
print(f"{'지표':<20}{'값'}")
print("-" * 78)
print(f"{'처리량':<20}{n_requests/total_time:.1f} req/s")
print(f"{'평균':<20}{lat.mean():.2f} ms")
print(f"{'p50':<20}{np.percentile(lat, 50):.2f} ms")
print(f"{'p95':<20}{np.percentile(lat, 95):.2f} ms")
print(f"{'p99':<20}{np.percentile(lat, 99):.2f} ms")
print(f"{'최댓값':<20}{lat.max():.2f} ms")
print(f"{'성공률':<20}{sum(1 for s in statuses if s == 200)/len(statuses)*100:.1f}%")
print("-" * 78)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

ax = axes[0]
ax.hist(lat, bins=40, color="#1E40AF", edgecolor="white")
for pct, color, label in [(50, "#0D9488", "p50"), (95, "#EA580C", "p95"),
                          (99, "#DC2626", "p99")]:
    v = np.percentile(lat, pct)
    ax.axvline(v, color=color, linestyle="--", linewidth=2,
               label=f"{label}={v:.1f}ms")
ax.set_xlabel("지연 (ms)")
ax.set_ylabel("요청 수")
ax.set_title("지연 분포")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axes[1]
window = 20
rolling = [lat[max(0, i-window):i+1].mean() for i in range(len(lat))]
ax.plot(rolling, linewidth=2, color="#0D9488")
ax.set_xlabel("요청 순번")
ax.set_ylabel(f"이동 평균 지연 ({window}건)")
ax.set_title("시간에 따른 안정성")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print()
print("[실무의 부하 시험 도구]")
print("  locust  : Python 으로 시나리오 작성")
print("  k6      : JavaScript, 가볍고 빠름")
print("  wrk     : 명령줄, 단순 부하")
print()
print("  이 장의 시험은 단일 프로세스이므로 참고용이다.")
print("  실제로는 별도 머신에서 부하를 걸어야 한다.")

---

## 9. 정리

### 만든 것

```
FastAPI 앱
├── /health          상태 확인
├── /answer          질문 처리 (Pydantic 검증)
├── /answer/stream   스트리밍 응답
└── /metrics         지표 조회

+ 미들웨어로 모든 요청의 지표 수집
+ Dockerfile 로 패키징
```

### 기억할 것

| 항목 | 요점 |
|---|---|
| Pydantic | 타입 힌트만으로 **검증 자동** |
| 자동 문서 | `/docs`에서 바로 시험 가능 |
| 오류 처리 | 내부 메시지 노출 금지, 상태 코드 구분 |
| 스트리밍 | TTFT 개선 — 체감 속도가 크게 다름 |
| **`async def` 안에서** | **`time.sleep()` 금지** → `await asyncio.sleep()` |
| 미들웨어 | 지표·로깅·인증을 일괄 처리 |
| Docker | requirements 먼저 COPY (캐시 활용) |
| 비밀 | 이미지가 아니라 **환경변수로** |
| 부하 시험 | 예상 최대의 2배로 |

### 이 책의 기술이 여기서 만난다

| 장 | 이 장에서 |
|---|---|
| 28장 RAG | API 로 감싼 대상 |
| 43장 프로젝트 A | 서비스로 만들 대상 |
| 37장 Agent | 도구 호출도 같은 방식으로 API 화 |
| 39장 평가 | 배포 전 회귀 시험 |
| 41장 서빙 | p95, TTFT, 워커 수 |

---

## 마치며

**실습에서 되는 것과 서비스로 쓸 수 있는 것은 다르다.**

이 장에서 그 간극을 메우는 것들을 다뤘다 —
입력 검증, 오류 처리, 동시성, 지표 수집, 패키징.

**어느 것도 모델 성능을 높이지 않는다.**
하지만 이것들 없이는 아무리 좋은 모델도 서비스가 되지 못한다.

### 다음 장

**41. LLM Serving과 LLMOps** — 이론편 25.6절. 만든 것을 실제로 운영하는 문제를 다룬다.
처리량·지연·비용을 어떻게 관리하는지 확인한다.